# Example 6.2 / 1: A Parameter Sweep

Companion notebook for **6.2 for Loops** in the AMPL lab Documentation, section "A Parameter Sweep". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Parameter Sweep

The model chooses production under a capacity limit. The script solves the same instance at
three capacities and records the objective value. In the Python tabs on this page and the rest
of this chapter, `ampl` is an `AMPL` object that has already read `model.mod`. `ampl.eval()` is
used because a compound AMPL loop has no direct entity method.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param capacity >= 0 default 0;            # available capacity, set by the sweep
param profit_per_unit > 0 default 4;      # profit per unit made
var Make >= 0;                            # units made
maximize Profit: profit_per_unit * Make;  # total profit
subject to Capacity: Make <= capacity;    # production cannot exceed capacity
set TRIALS = 10..30 by 10;                # capacities to try
param trial_profit {TRIALS} default 0;    # objective value recorded for each capacity

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["profit_per_unit"] = 4

In [5]:
# run.py
ampl.option["solver"] = "highs"
ampl.eval("""
for {c in TRIALS} {
    let capacity := c;
    solve;
    let trial_profit[c] := Profit;
}
""")
for capacity, profit in ampl.param["trial_profit"].to_dict().items():
    print(f"{int(capacity)}: {profit:.0f}")

HiGHS 1.14.0: optimal solution; objective 40
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0: optimal solution; objective 80
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0: optimal solution; objective 120
0 simplex iterations
0 barrier iterations
10: 40
20: 80
30: 120


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param profit_per_unit := 4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
for {c in TRIALS} {
    let capacity := c;
    solve;
    let trial_profit[c] := Profit;
}
display trial_profit;

HiGHS 1.14.0: optimal solution; objective 40
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0: optimal solution; objective 80
0 simplex iterations
0 barrier iterations
HiGHS 1.14.0: optimal solution; objective 120
0 simplex iterations
0 barrier iterations
trial_profit [*] :=
10   40
20   80
30  120
;

